# Day 09. Exercise 04
# Pipelines and OOP

## 0. Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from tqdm import tqdm
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import ParameterGrid, cross_val_score
from joblib import load


## 1. Preprocessing pipeline

Create three custom transformers, the first two out of which will be used within a [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html).

1. `FeatureExtractor()` class:
 - Takes a dataframe with `uid`, `labname`, `numTrials`, `timestamp` from the file [`checker_submits.csv`](https://drive.google.com/file/d/14voc4fNJZiLEFaZyd8nEG-lQt5JjatYw/view?usp=sharing).
 - Extracts `hour` from `timestamp`.
 - Extracts `weekday` from `timestamp` (numbers).
 - Drops the `timestamp` column.
 - Returns the new dataframe.


2. `MyOneHotEncoder()` class:
 - Takes the dataframe from the result of the previous transformation and the name of the target column.
 - Identifies all the categorical features and transforms them with `OneHotEncoder()`. If the target column is categorical too, then the transformation should not apply to it.
 - Drops the initial categorical features.
 - Returns the dataframe with the features and the series with the target column.


3. `TrainValidationTest()` class:
 - Takes `X` and `y`.
 - Returns `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` (`test_size=0.2`, `random_state=21`, `stratified`).


In [2]:
class FeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X, y=None):
        new_X = X.copy()
        new_X['timestamp'] = pd.to_datetime(X['timestamp'])        
        new_X['hour'] = new_X['timestamp'].dt.hour
        new_X['weekday'] = new_X['timestamp'].dt.weekday
        new_X = new_X.drop('timestamp', axis=1)
        return new_X


In [3]:

class MyOneHotEncoder:
    def __init__(self, target_column=None):
        self.target_column = target_column
        self.encoder = OneHotEncoder()
        self.cat_cols = None

    def fit(self, df, y=None):
        cat_cols = df.select_dtypes(include=["object"]).columns

        if self.target_column is not None and self.target_column in cat_cols:
            cat_cols = cat_cols.drop(self.target_column)

        self.cat_cols = list(cat_cols)
        self.encoder.fit(df[self.cat_cols])

        return self

    def transform(self, df):
        ohe_array = self.encoder.transform(df[self.cat_cols]).toarray()

        ohe_columns = self.encoder.get_feature_names_out(self.cat_cols)
        ohe_df = pd.DataFrame(ohe_array, columns=ohe_columns, index=df.index)

        result = pd.concat([df, ohe_df], axis=1)

        result = result.drop(columns=self.cat_cols)

        return result


In [4]:
class TrainValidationTest:
 
    def __init__(self, test_size=0.2, random_state=21):

        self.test_size = test_size
        self.random_state = random_state
    
    def split(self, X, y):

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=self.test_size,
            random_state=self.random_state,
            stratify=y
        )

        validation_size = self.test_size / (1 - self.test_size)
        
        X_train, X_valid, y_train, y_valid = train_test_split(
            X_temp, y_temp,
            test_size=validation_size,
            random_state=self.random_state,
            stratify=y_temp
        )
        
        return X_train, X_valid, X_test, y_train, y_valid, y_test


In [5]:
#Data Preparation
data = pd.read_csv('../../data/checker_submits.csv')
df = pd.DataFrame(data)

feature_extractor = FeatureExtractor()
new_df = feature_extractor.fit_transform(df)

encoder = MyOneHotEncoder()
encoder.fit(new_df)
encoded_df = encoder.transform(new_df)

X = encoded_df.drop('weekday', axis=1)
y = encoded_df['weekday']
train_val = TrainValidationTest(test_size=0.2, random_state=21)
X_train, X_valid, X_test, y_train, y_valid, y_test = train_val.split(X, y)

## 2. Model selection pipeline

`ModelSelection()` class

 - Takes a list of `GridSearchCV` instances and a dict where the keys are the indexes from that list and the values are the names of the models, the example is below in the reverse order (from high-level to low-level perspective):

```
ModelSelection(grids, grid_dict)

grids = [gs_svm, gs_tree, gs_rf]

gs_svm = GridSearchCV(estimator=svm, param_grid=svm_params, scoring='accuracy', cv=2, n_jobs=jobs), where jobs you can specify by yourself

svm_params = [{'kernel':('linear', 'rbf', 'sigmoid'), 'C':[0.01, 0.1, 1, 1.5, 5, 10], 'gamma': ['scale', 'auto'], 'class_weight':('balanced', None), 'random_state':[21], 'probability':[True]}]
```

 - Method `choose()` takes `X_train`, `y_train`, `X_valid`, `y_valid` and returns the name of the best classifier among all the models on the validation set
 - Method `best_results()` returns a dataframe with the columns `model`, `params`, `valid_score` where the rows are the best models within each class of models.

```
model	params	valid_score
0	SVM	{'C': 10, 'class_weight': None, 'gamma': 'auto...	0.772727
1	Decision Tree	{'class_weight': 'balanced', 'criterion': 'gin...	0.801484
2	Random Forest	{'class_weight': None, 'criterion': 'entropy',...	0.855288
```

 - When you iterate through the parameters of a model class, print the name of that class and show the progress using `tqdm.notebook`, in the end of the cycle print the best model of that class.

```
Estimator: SVM
100%
125/125 [01:32<00:00, 1.36it/s]
Best params: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf', 'probability': True, 'random_state': 21}
Best training accuracy: 0.773
Validation set accuracy score for best params: 0.878 

Estimator: Decision Tree
100%
57/57 [01:07<00:00, 1.22it/s]
Best params: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 21, 'random_state': 21}
Best training accuracy: 0.801
Validation set accuracy score for best params: 0.867 

Estimator: Random Forest
100%
284/284 [06:47<00:00, 1.13s/it]
Best params: {'class_weight': None, 'criterion': 'entropy', 'max_depth': 22, 'n_estimators': 50, 'random_state': 21}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.907 

Classifier with best validation set accuracy: Random Forest
```

In [6]:


class ModelSelection:
    def __init__(self, grids, grid_dict):

        self.grids = grids
        self.grid_dict = grid_dict
        self.results = []

        self.best_model_name = None
        self.best_valid_score = -np.inf
        self.best_estimator = None
        self.best_estimator_params = None
        
    def choose(self, X_train, y_train, X_valid, y_valid):
        self.best_model_name = None
        self.best_valid_score = -np.inf
        self.best_estimator = None
        self.best_estimator_params = None
        
        for idx, grid in enumerate(self.grids):
            model_name = self.grid_dict[idx]
            print(f"\nEstimator: {model_name}")
            
            param_grid = grid.param_grid
            estimator = grid.estimator
            cv = grid.cv
            scoring = grid.scoring
            
            param_combinations = list(ParameterGrid(param_grid))
            
            best_params = None
            best_train_score = -np.inf
            best_estimator = None
            
            for params in tqdm(param_combinations, total=len(param_combinations)):
                model = estimator.__class__(**params)
                
                cv_scores = cross_val_score(
                    model, X_train, y_train, cv=cv, scoring=scoring
                )
                train_score = cv_scores.mean()
                
                if train_score > best_train_score:
                    best_train_score = train_score
                    best_params = params
                    best_estimator = estimator.__class__(**params)
                    best_estimator.fit(X_train, y_train)
            
            valid_score = best_estimator.score(X_valid, y_valid)
            
            self.results.append({
                'model': model_name,
                'params': best_params,
                'train_score': best_train_score,
                'valid_score': valid_score,
                'estimator': best_estimator,
            })
            
            print(f"Best params: {best_params}")
            print(f"Best training accuracy: {best_train_score:.3f}")
            print(f"Validation set accuracy score for best params: {valid_score:.3f}")
            print()
            
            if valid_score > self.best_valid_score:
                self.best_valid_score = valid_score
                self.best_model_name = model_name
                self.best_estimator = best_estimator
                self.best_estimator_params = best_params
        
        print(f'Classifier with best validation set accuracy: {self.best_model_name}')
        return self.best_model_name
    
    def best_results(self):
     
        df = pd.DataFrame(self.results)
        return df[['model', 'params', 'valid_score']].reset_index(drop=True)
    
    
    def get_config_dict(self):
 
        return {
            "model_class": self.best_estimator.__class__,
            "params": self.best_estimator_params,
        }


In [7]:

#GRID
svc = SVC(random_state=21, probability=True)

svc_param_grid = {
    'kernel': ['linear'],
    'C': [0.01,10],
    'gamma':['auto'],
    'class_weight': ['balanced']
}
svc_grid = GridSearchCV(estimator=svc, 
                    param_grid=svc_param_grid,
                    cv=2, 
                    scoring='accuracy',
                    n_jobs=-1)
#
tree = DecisionTreeClassifier(random_state=21)
tree_param_grid = {
    'max_depth': (24,25),
    'class_weight': [None],
    'criterion': ['gini']
}
tree_grid = GridSearchCV(estimator=tree, 
                    param_grid=tree_param_grid, 
                    cv=2,
                    scoring='accuracy',
                    n_jobs=-1)

#
forest = RandomForestClassifier(random_state=21)

forest_params = {
    'n_estimators': [100],
    'max_depth': [49,50],          
    'class_weight': ['balanced'],
    'criterion': ['entropy']
}

forest_grid = GridSearchCV(
    estimator=forest,
    param_grid=forest_params,
    cv=2,
    scoring='accuracy',
    n_jobs=-1
)


grids = [svc_grid, tree_grid,forest_grid ]

grid_dict = {0: "SVM", 1: "Decision Tree", 2: "Random Forest"}

# model_selector = ModelSelection(grids, grid_dict)
# best_model_name =model_selector.choose(X_train, y_train, X_valid, y_valid)

# results_df = pd.DataFrame(model_selector.best_results())
# config_dict = model_selector.get_config_dict()

# model_instance = config_dict["model_class"](**config_dict["params"])
# model_instance.set_params(random_state=21)


## 3. Finalization

`Finalize()` class
 - Takes an estimator.
 - Method `final_score()` takes `X_train`, `y_train`, `X_test`, `y_test` and returns the accuracy of the model as in the example below:
```
final.final_score(X_train, y_train, X_test, y_test)
Accuracy of the final model is 0.908284023668639
```
 - Method `save_model()` takes a path, saves the model to this path and prints that the model was successfully saved.

In [8]:
from sklearn.metrics import accuracy_score
from joblib import dump

class Finalize():
    def __init__(self, model):
        self._model = model

    def final_score(self, X_train, y_train, X_test, y_test):
        fitted_model = self._model.fit(X_train, y_train)
        predict = fitted_model.predict(X_test)
        acc = accuracy_score(y_test, predict)
        return f'Accuracy of the final model is {acc}'

    def save_model(self, path):
        dump(self._model, path)
        print('The model was successfully saved')


## 4. Main program

1. Load the data from the file (****name of file****).
2. Create the preprocessing pipeline that consists of two custom transformers: `FeatureExtractor()` and `MyOneHotEncoder()`:
```
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
```
3. Use that pipeline and its method `fit_transform()` on the initial dataset.
```
data = preprocessing.fit_transform(df)
```
4. Get `X_train`, `X_valid`, `X_test`, `y_train`, `y_valid`, `y_test` using `TrainValidationTest()` and the result of the pipeline.
5. Create an instance of `ModelSelection()`, use the method `choose()` applying it to the models that you want and parameters that you want, get the dataframe of the best results.
6. create an instance of `Finalize()` with your best model, use method `final_score()` and save the model in the format: `name_of_the_model_{accuracy on test dataset}.sav`.

That is it, congrats!

In [9]:
preprocessing = Pipeline([('feature_extractor', FeatureExtractor()), ('onehot_encoder', MyOneHotEncoder('dayofweek'))])
data = preprocessing.fit_transform(df)
train_val = TrainValidationTest()
X = data.drop('weekday', axis=1)
y = data['weekday']
X_train, X_valid, X_test, y_train, y_valid, y_test = train_val.split(X, y)
model_selector = ModelSelection(grids, grid_dict)
best_model_name = model_selector.choose(X_train, y_train, X_valid, y_valid)

results_df = pd.DataFrame(model_selector.best_results())
config_dict = model_selector.get_config_dict()

model_instance = config_dict["model_class"](**config_dict["params"])
model_instance.set_params(random_state=21)
print(results_df)



Estimator: SVM


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:39<00:00, 19.64s/it]


Best params: {'C': 10, 'class_weight': 'balanced', 'gamma': 'auto', 'kernel': 'linear'}
Best training accuracy: 0.703
Validation set accuracy score for best params: 0.712


Estimator: Decision Tree


100%|██████████| 2/2 [00:00<00:00, 60.30it/s]


Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 24}
Best training accuracy: 0.809
Validation set accuracy score for best params: 0.869


Estimator: Random Forest


100%|██████████| 2/2 [00:01<00:00,  1.09it/s]

Best params: {'class_weight': 'balanced', 'criterion': 'entropy', 'max_depth': 49, 'n_estimators': 100}
Best training accuracy: 0.855
Validation set accuracy score for best params: 0.872

Classifier with best validation set accuracy: Random Forest
           model                                             params  \
0            SVM  {'C': 10, 'class_weight': 'balanced', 'gamma':...   
1  Decision Tree  {'class_weight': None, 'criterion': 'gini', 'm...   
2  Random Forest  {'class_weight': 'balanced', 'criterion': 'ent...   

   valid_score  
0     0.712166  
1     0.869436  
2     0.872404  


In [10]:
#Finalizing
finalizer = Finalize(model_instance)

final_score = finalizer.final_score(X_train, y_train, X_test, y_test)
print(final_score)
model_filename = f'{best_model_name}_{final_score.split()[-1]}.sav'
finalizer.save_model(model_filename)

Accuracy of the final model is 0.908284023668639
The model was successfully saved


In [11]:
final_test = load(model_filename)
finale = Finalize(final_test)
finale.final_score(X_train, y_train, X_test, y_test)

'Accuracy of the final model is 0.908284023668639'